In [2]:
import sentence_transformers
import faiss
import numpy as np

print("sentence-transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("NumPy:", np.__version__)

sentence-transformers: 6.0.0
FAISS: 1.15.0
NumPy: 2.5.2


# Day 2 — Mini Semantic Search Engine using FAISS

## Objective

Build a mini semantic search engine using Python and FAISS.

The system converts knowledge base sentences into embeddings, stores them in a FAISS vector index, and retrieves the most semantically similar sentences for a user query.

This simulates the retrieval component used in real-world Retrieval-Augmented Generation (RAG) systems.

## Task 1 — Setup & Embedding Generation

A small knowledge base containing customer-support-related sentences is created.

The sentences cover topics such as password reset, billing, account management, login issues, refunds, and subscriptions.

Each sentence is converted into a numerical embedding using the `all-MiniLM-L6-v2` sentence-transformer model.

In [3]:
knowledge_base = [
    "You can reset your password by clicking the Forgot Password link on the login page.",
    "To update your billing information, go to Account Settings and select Payment Methods.",
    "If you cannot log in, check that your email address and password are correct.",
    "You can update your account email address from the Account Settings page.",
    "Refund requests can be submitted through the Billing section of your account.",
    "To cancel your subscription, open Account Settings and select Subscription.",
    "Your monthly invoice can be downloaded from the Billing section.",
    "If your account is locked, contact customer support to regain access.",
    "You can change your account password from the Security Settings page.",
    "Payment failures may occur if your card has expired or has insufficient funds."
]

print("Number of knowledge base sentences:", len(knowledge_base))

Number of knowledge base sentences: 10


## Generate Embeddings

The `all-MiniLM-L6-v2` model converts each knowledge base sentence into a 384-dimensional vector.

Therefore, with 10 sentences, the resulting embedding matrix should have the shape:

`(10, 384)`

In [5]:
import os

SYSTEM_CA = "/etc/ssl/certs/ca-certificates.crt"

os.environ["REQUESTS_CA_BUNDLE"] = SYSTEM_CA
os.environ["SSL_CERT_FILE"] = SYSTEM_CA
os.environ["CURL_CA_BUNDLE"] = SYSTEM_CA

print("Using certificate bundle:", SYSTEM_CA)

Using certificate bundle: /etc/ssl/certs/ca-certificates.crt


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(knowledge_base)

print("Embedding matrix shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding matrix shape: (10, 384)


## Task 2 — Build a FAISS Index

FAISS is used as the vector store for the knowledge base embeddings.

We create an `IndexFlatL2` index with 384 dimensions because `all-MiniLM-L6-v2` generates 384-dimensional embeddings.

The embeddings are normalized before being added to FAISS. With normalized vectors, L2 distance can be used to obtain cosine-similarity-like rankings.

In [7]:
import faiss
import numpy as np

embedding_matrix = np.array(
    embeddings,
    dtype="float32"
)

print("Before normalization:", embedding_matrix.shape)

Before normalization: (10, 384)


In [8]:
faiss.normalize_L2(embedding_matrix)

In [9]:
dimension = 384

index = faiss.IndexFlatL2(dimension)

index.add(embedding_matrix)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 10


## Task 3 — Semantic Search with FAISS

A user query is converted into an embedding using the same `all-MiniLM-L6-v2` model.

The query embedding is normalized and searched against the FAISS index.

The top 3 most similar knowledge base sentences are retrieved.

In [10]:
def semantic_search(query, k=3):
    
    # Convert query into embedding
    query_embedding = model.encode([query])
    
    # Convert to float32
    query_embedding = np.array(
        query_embedding,
        dtype="float32"
    )
    
    # Normalize query embedding
    faiss.normalize_L2(query_embedding)
    
    # Search FAISS
    distances, indices = index.search(
        query_embedding,
        k
    )
    
    print(f"\nQuery: {query}")
    print("\nRank | Score | Matched Sentence")
    print("-" * 80)
    
    for rank, (distance, idx) in enumerate(
        zip(distances[0], indices[0]),
        start=1
    ):
        print(
            f"{rank:<4} | "
            f"{distance:.4f} | "
            f"{knowledge_base[idx]}"
        )

In [11]:
semantic_search("I forgot my password")


Query: I forgot my password

Rank | Score | Matched Sentence
--------------------------------------------------------------------------------
1    | 0.5572 | You can reset your password by clicking the Forgot Password link on the login page.
2    | 0.8423 | If your account is locked, contact customer support to regain access.
3    | 0.8852 | You can change your account password from the Security Settings page.


In [12]:
semantic_search("My credit card payment failed")


Query: My credit card payment failed

Rank | Score | Matched Sentence
--------------------------------------------------------------------------------
1    | 0.5549 | Payment failures may occur if your card has expired or has insufficient funds.
2    | 1.2791 | Refund requests can be submitted through the Billing section of your account.
3    | 1.2877 | To update your billing information, go to Account Settings and select Payment Methods.


In [13]:
semantic_search("How can I cancel my subscription?")


Query: How can I cancel my subscription?

Rank | Score | Matched Sentence
--------------------------------------------------------------------------------
1    | 0.1237 | To cancel your subscription, open Account Settings and select Subscription.
2    | 1.1311 | Refund requests can be submitted through the Billing section of your account.
3    | 1.2184 | To update your billing information, go to Account Settings and select Payment Methods.


## Task 4 — Interactive CLI

The semantic search engine is wrapped inside an interactive loop.

Users can continuously enter questions and receive the top 3 matching knowledge base sentences.

Typing `exit` terminates the program.

In [14]:
while True:
    
    query = input("\nEnter your query (type 'exit' to quit): ")
    
    if query.lower() == "exit":
        print("Exiting semantic search engine...")
        break
    
    semantic_search(query)


Query: I can't access my account

Rank | Score | Matched Sentence
--------------------------------------------------------------------------------
1    | 0.5093 | If your account is locked, contact customer support to regain access.
2    | 0.7318 | If you cannot log in, check that your email address and password are correct.
3    | 1.0344 | You can change your account password from the Security Settings page.
Exiting semantic search engine...
